# 🐚 Akoya Pearl Miner - Google Colab

**PENTING:** Runtime → Change runtime type → **L4 GPU** (T4 tidak didukung)

Kalau L4 tidak tersedia, coba A100. T4 TIDAK BISA (compute 7.5 terlalu rendah).

In [ ]:
#@title 1️⃣ Check GPU & Install Tools
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print(f'GPU: {gpu}')
cc = gpu.split(',')[1].strip()
major = int(cc.split('.')[0])
if major < 8:
    print('\n❌ GPU compute < 8.0 — Akoya miner TIDAK SUPPORT GPU ini!')
    print('   Ganti ke L4 atau A100: Runtime → Change runtime type → L4')
else:
    print(f'\n✅ GPU supported (compute {cc})')
    !apt-get update -qq && apt-get install -y -qq skopeo umoci > /dev/null 2>&1
    print('✅ Tools installed')

In [ ]:
#@title 2️⃣ Download & Setup Miner
import os, shutil, subprocess

MINER_DIR = '/content/akoya-miner'
for d in [MINER_DIR, '/content/akoya-oci', '/content/akoya-bundle']:
    if os.path.exists(d): shutil.rmtree(d)

print('Downloading Akoya miner...')
!skopeo copy docker://registry.akoyapool.com/akoya-miner:latest oci:/content/akoya-oci:latest 2>&1 | tail -3
!umoci unpack --image /content/akoya-oci:latest /content/akoya-bundle 2>&1 | tail -1

os.makedirs(MINER_DIR, exist_ok=True)
!cp -r /content/akoya-bundle/rootfs/app/* {MINER_DIR}/
!chmod +x {MINER_DIR}/akoya-miner

# Auto-select kernel
cc = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip().split('\n')[0]
major, minor = cc.split('.')
lib_dir = f'{MINER_DIR}/lib'

if int(major) == 12: src = 'blackwell'
elif int(major) == 9: src = 'h100'
elif int(major) == 8 and int(minor) == 9: src = 'ada'
elif int(major) == 8: src = 'ampere'
else: src = 'portable'

target = f'{lib_dir}/libpearl_gemm_capi.so'
if os.path.lexists(target): os.unlink(target)
os.symlink(f'{lib_dir}/libpearl_gemm_capi_{src}.so', target)

print(f'\n✅ Ready! Kernel: {src} (compute {cc})')

In [ ]:
#@title 3️⃣ START MINING 🚀
import subprocess, os, signal, time

MINER_DIR = '/content/akoya-miner'
WALLET = 'prl1pq4v3a3677vymqt47jg6qvcxhgavrldmu4xe2k9hlx95lqqht2fyssutugf'
WORKER = 'colab-gpu'

env = os.environ.copy()
env.update({
    'AKOYA_POOL_WALLET': WALLET,
    'AKOYA_POOL_WORKER': WORKER,
    'AKOYA_POOL_HOST': 'pool-v2.akoyapool.com',
    'AKOYA_POOL_PORT': '443',
    'AKOYA_POOL_USE_TLS': '1',
    'AKOYA_GPU_INDICES': 'all',
    'AKOYA_METRICS_PORT': '9100',
    'AKOYA_PEARL_GEMM_LIB': f'{MINER_DIR}/lib/libpearl_gemm_capi.so',
    'AKOYA_PEARL_MINING_LIB': f'{MINER_DIR}/lib/libpearl_mining_capi.so',
    'LD_LIBRARY_PATH': f"{MINER_DIR}/lib:" + os.environ.get('LD_LIBRARY_PATH', ''),
})
os.makedirs('/var/lib/akoya-miner', exist_ok=True)

print(f'🐚 Akoya Pearl Miner')
print(f'   Wallet: {WALLET[:15]}...{WALLET[-8:]}')
print(f'   Worker: {WORKER}')
print('='*60)

proc = subprocess.Popen(
    [f'{MINER_DIR}/akoya-miner', 'mine-blocks'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=env,
)

try:
    start = time.time()
    while proc.poll() is None:
        line = proc.stdout.readline()
        if line: print(line.rstrip(), flush=True)
    out = proc.stdout.read()
    if out: print(out)
    if proc.returncode != 0:
        print(f'\n❌ Miner exited with code {proc.returncode}')
        if proc.returncode == -11:
            print('SEGFAULT — GPU kernel tidak kompatibel.')
            print('Ganti runtime ke L4 atau A100: Runtime → Change runtime type')
except KeyboardInterrupt:
    proc.send_signal(signal.SIGINT)
    proc.wait(timeout=30)
    print(f'\n✅ Stopped after {int(time.time()-start)}s')